# **Import**

In [1]:
import pickle
import numpy as np 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

2026-02-08 12:09:20.410939: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770552560.620904      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770552560.683495      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770552561.205777      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770552561.205822      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770552561.205828      55 computation_placer.cc:177] computation placer alr

# **Load Pickel File**

In [20]:
with open("/kaggle/input/wiki-text-2-vocab/word_to_index.pkl", "rb") as file:
    word_to_index = pickle.load(file)
with open("/kaggle/input/wiki-text-2-vocab/index_to_word.pkl", "rb") as file:
    index_to_file = pickle.load(file)
with open("/kaggle/input/wiki-text-2-vocab/encoded_text.pkl", "rb") as file:
    encoded_text = pickle.load(file)

# **Sliding Window Implementation**

In [21]:
vocab_size = len(word_to_index) + 1 

def sliding_window_sequence_dataset(token_list, sequence_length):
    x = []
    y = []

    for i in range(sequence_length, len(token_list)):
        x.append(token_list[i - sequence_length : i])
        y.append(token_list[i])

    return np.array(x), np.array(y)

In [22]:
sequence_length = 20
x, y = sliding_window_sequence_dataset(encoded_text, sequence_length)
print(f"x shape : {x.shape}")
print(f"y shape : {y.shape}")

x shape : (2393391, 20)
y shape : (2393391,)


# **LSTM**

In [23]:
x = x.astype("int32")
y = y.astype("int32")

model = Sequential([
    Embedding(input_dim = vocab_size, output_dim = 100),
    LSTM(150),
    Dense(vocab_size, activation="softmax")
])

In [24]:
model.compile(
    loss = "sparse_categorical_crossentropy",
    optimizer = "adam",
    metrics = ['accuracy'] 
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# ***Training***

In [25]:
history = model.fit(
    x,
    y,
    epochs = 80,
    batch_size = 256,
    validation_split = 0.2
)

Epoch 1/80


I0000 00:00:1770552657.654386     123 cuda_dnn.cc:529] Loaded cuDNN version 91002


7480/7480 ━━━━━━━━━━━━━━━━━━━━ 153s 20ms/step - accuracy: 0.1289 - loss: 6.5869 - val_accuracy: 0.1866 - val_loss: 6.0223
Epoch 2/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.1980 - loss: 5.5774 - val_accuracy: 0.2028 - val_loss: 5.7690
Epoch 3/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2193 - loss: 5.1370 - val_accuracy: 0.2105 - val_loss: 5.6876
Epoch 4/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2357 - loss: 4.8471 - val_accuracy: 0.2148 - val_loss: 5.6749
Epoch 5/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2480 - loss: 4.6351 - val_accuracy: 0.2170 - val_loss: 5.7033
Epoch 6/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2586 - loss: 4.4664 - val_accuracy: 0.2174 - val_loss: 5.7410
Epoch 7/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2691 - loss: 4.3224 - val_accuracy: 0.2179 - val_loss: 5.7984
Epoch 8/80
7480/7480 ━━━━━━━━━━━━━━━━━━━━ 150s 20ms/step - accuracy: 0.2780 - lo

In [26]:
model.save("word_pred_LSTM_model.keras")